In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Warnings
import warnings
warnings.filterwarnings('ignore')

In [42]:
# Loading user_item matrix
rat_mat = np.load("../transformed_Data/user_item_matrix.npy")
# Loading user indices correspondance
user_corres = np.load("../transformed_Data/user_corres_matrix_index.npy")
# Loading movie indices correspondance
movie_corres = np.load("../transformed_Data/movie_corres_matrix_index.npy")
# loading correspondance between movie indices and movie titles
movies_titles = pd.read_csv("../transformed_Data/MovieID_title.csv")
# loading ratings
ratings = pd.read_csv("../transformed_Data/ratings_small.csv")

In [3]:
# sparsity of the ratings matrix:
sparsity = 1.0 - ( np.count_nonzero(rat_mat) / float(rat_mat.size) )
print(sparsity)

0.9431750381059136


In [45]:
ratings.head()

,UserID,MovieID,Rating
0,1,1193,5
1,1,661,3
2,1,914,3
3,1,3408,4
4,1,2355,5


In [47]:
n_users = len(ratings.iloc[:,0].unique())
n_items = len(ratings.iloc[:,1].unique())

print("nb of users :", n_users)
print("nb of movies :", n_items)

nb of users : 518
nb of movies : 3255


### 1. Computation of cos similarity matrix and predictions

#### 1.1 Prediction function

In [4]:
#cos similarity between user and items
from sklearn.metrics.pairwise import pairwise_distances
user_similarity = pairwise_distances(rat_mat, metric='cosine')
item_similarity = pairwise_distances(rat_mat.T, metric='cosine')

In [ ]:
def predict(ratings,n_users, n_items, similarity, type='user'):



    if type == 'user':
        
        mean_user_rating = ratings.mean(axis=1)#mean for each user (type = float)
        #np.newaxis to convert mean_user_rating from 1D array of float to 2D array in order to user it with ratings
        #then normalization of ratings var (rating - E)
        ratings_diff = (ratings - mean_user_rating[:, np.newaxis]) #(type === 2D array like rating var)
        pred = mean_user_rating[:, np.newaxis] + similarity.dot(ratings_diff) / np.array([np.abs(similarity).sum(axis=1)]).T
       # pred = ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)]).T 

    elif type == 'item':
        pred = ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)]) 
        
    x = np.zeros((n_users, n_items))
    for i in range(0,n_items):
        a=max(pred[:,i])
        b=min(pred[:,i])
        c=0
        d=5
        for j in range(0,n_users):
            x[j,i]=(pred[:,i][j]-(a-c))*d/(b-a+c)
    
    return x

In [41]:
user_prediction = predict(rat_mat, user_similarity, type='user')
#item_prediction = predict(rat_mat, item_similarity, type='item')

nb of users : 518
nb of movies : 518


#### 1.2 transformation of indices in recommandations to movie titles

In [7]:
movie_corres.shape

(3255, 2)

In [8]:
movie_corres[:10, :]

array([[ 0.,  1.],
       [ 1.,  2.],
       [ 2.,  3.],
       [ 3.,  4.],
       [ 4.,  5.],
       [ 5.,  6.],
       [ 6.,  7.],
       [ 7.,  8.],
       [ 8.,  9.],
       [ 9., 10.]])

In [12]:
movie_corres[-10:, :]

array([[3245., 3943.],
       [3246., 3944.],
       [3247., 3945.],
       [3248., 3946.],
       [3249., 3947.],
       [3250., 3948.],
       [3251., 3949.],
       [3252., 3950.],
       [3253., 3951.],
       [3254., 3952.]])

In [9]:
movies_titles.shape

(3883, 2)

In [10]:
movies_titles.head(10)

,MovieID,title
0,1,Toy Story
1,2,Jumanji
2,3,Grumpier Old Men
3,4,Waiting to Exhale
4,5,Father of the Bride Part II
5,6,Heat
6,7,Sabrina
7,8,Tom and Huck
8,9,Sudden Death
9,10,GoldenEye


In [13]:
movies_titles.tail(10)

,MovieID,title
3873,3943,Bamboozled
3874,3944,Bootmen
3875,3945,Digimon: The Movie
3876,3946,Get Carter
3877,3947,Get Carter
3878,3948,Meet the Parents
3879,3949,Requiem for a Dream
3880,3950,Tigerland
3881,3951,Two Family House
3882,3952,"Contender, The"


In [18]:
# merge between movie_corres (array of correspondance between item index in prediction matrix and titles)
movie_corres_df = pd.DataFrame(data = movie_corres.astype(np.int32), columns=["index", "MovieID"])

In [19]:
movie_corres_df.head()

,index,MovieID
0,0,1
1,1,2
2,2,3
3,3,4
4,4,5


In [20]:
index_title = movie_corres_df.merge(movies_titles, on="MovieID")

In [21]:
index_title.shape

(3255, 3)

In [22]:
index_title.head()

,index,MovieID,title
0,0,1,Toy Story
1,1,2,Jumanji
2,2,3,Grumpier Old Men
3,3,4,Waiting to Exhale
4,4,5,Father of the Bride Part II


In [23]:
index_title.tail()

,index,MovieID,title
3250,3250,3948,Meet the Parents
3251,3251,3949,Requiem for a Dream
3252,3252,3950,Tigerland
3253,3253,3951,Two Family House
3254,3254,3952,"Contender, The"


#### 1.3 analysis of user n°0

In [25]:
# number of movies ratde by user 0
np.count_nonzero(rat_mat[0, :])

53

In [29]:
user_0 = pd.DataFrame(data = rat_mat[0, :].T, columns=["rating"])
user_0.reset_index(inplace=True)
user_0.head()

,index,rating
0,0,5.0
1,1,0.0
2,2,0.0
3,3,0.0
4,4,0.0


In [30]:
user_0.shape

(3255, 2)

In [31]:
user_0 = user_0.merge(index_title, on="index")
user_0.head()

,index,rating,MovieID,title
0,0,5.0,1,Toy Story
1,1,0.0,2,Jumanji
2,2,0.0,3,Grumpier Old Men
3,3,0.0,4,Waiting to Exhale
4,4,0.0,5,Father of the Bride Part II


In [37]:
min_rat = 5.0
user_movies = user_0[user_0["rating"] >= min_rat]
print("number of movies with a minimum rating of", str(min_rat), " : ", len(user_movies))
user_movies.head(len(user_movies))

number of movies with a minimum rating of 5.0  :  18


,index,rating,MovieID,title
0,0,5.0,1,Toy Story
46,46,5.0,48,Pocahontas
131,131,5.0,150,Apollo 13
475,475,5.0,527,Schindler's List
531,531,5.0,595,Beauty and the Beast
829,829,5.0,1022,Cinderella
835,835,5.0,1028,Mary Poppins
836,836,5.0,1029,Dumbo
842,842,5.0,1035,"Sound of Music, The"
954,954,5.0,1193,One Flew Over the Cuckoo's Nest


In [39]:
user_prediction.shape

(518, 518)